# Cord Blood CITE-seq — Seurat WNN

BD Rhapsody 2.2.1, Targeted Immune Response Panel (389 genes) + 37 AbSeq, SMK 6 donors, single cartridge.

In [ ]:
# # Installation 
# if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("glmGamPoi", "SingleR", "celldex", "scDblFinder", "SingleCellExperiment"))
# install.packages(c("Seurat", "sctransform", "SoupX", "harmony", "patchwork",
#                    "dplyr", "ggplot2", "Matrix", "future", "writexl"))
# devtools::install_github('immunogenomics/presto')   # speeds up FindAllMarkers

In [1]:
suppressPackageStartupMessages({
  library(Seurat); library(SeuratObject)
  library(Matrix); library(SoupX)
  library(sctransform); library(harmony)
  library(ggplot2); library(patchwork); library(dplyr); library(future)
})

options(Seurat.object.assay.version = "v5")   # Seurat v5 assays
set.seed(42)

plan("multicore", workers = 10)
options(future.globals.maxSize = 16000 * 1024^2)  
future.seed <- TRUE

cat("Seurat", as.character(packageVersion("Seurat")), "\n")

Seurat 5.4.0 


## Preprocessing

In [ ]:
# paths
DATA_DIR <- "Data"
UNFILT   <- file.path(DATA_DIR, "Cord-Blood_RSEC_MolsPerCell_Unfiltered_MEX")
TAGCALLS <- file.path(DATA_DIR, "Cord-Blood_Sample_Tag_Calls.csv")
OUT_DIR  <- "results"; dir.create(OUT_DIR, showWarnings = FALSE)

# expected sizes (sanity check) 
EXPECT <- list(unfiltered = 681694L, called = 33630L, singlets = 22092L, features = 430L)

In [ ]:
full <- Read10X(UNFILT, gene.column = 2)      # list: Gene Expression / Antibody Capture

stopifnot(is.list(full), all(c("Gene Expression","Antibody Capture") %in% names(full)))
rna_all <- full[["Gene Expression"]]
adt_all <- full[["Antibody Capture"]]

cat(sprintf("RNA: %d features x %d barcodes\n", nrow(rna_all), ncol(rna_all)))
cat(sprintf("ADT: %d features x %d barcodes\n", nrow(adt_all), ncol(adt_all)))
stopifnot(ncol(rna_all) == EXPECT$unfiltered,
          nrow(rna_all) + nrow(adt_all) == EXPECT$features)

In [ ]:
# sample tag calls 

hdr   <- readLines(TAGCALLS, n = 30)
skip_n<- grep("^Cell_Index", hdr)[1] - 1
calls <- read.csv(TAGCALLS, skip = skip_n, stringsAsFactors = FALSE)
calls$Cell_Index <- as.character(calls$Cell_Index)

cat("calls:", nrow(calls), "\n")
print(table(calls$Sample_Name))

In [ ]:
# three barcode sets 
bc_all     <- colnames(rna_all)
bc_called  <- calls$Cell_Index                       
bc_singlet <- calls$Cell_Index[!calls$Sample_Tag %in% c("Multiplet","Undetermined")]
bc_empty   <- setdiff(bc_all, bc_called)             

stopifnot(all(bc_called %in% bc_all))

cat(sprintf("total barcodes  : %7d\n", length(bc_all)))
cat(sprintf("  called        : %7d \n", length(bc_called)))
cat(sprintf("  of which singlets: %5d\n", length(bc_singlet)))
cat(sprintf("  empty (soup)  : %7d\n", length(bc_empty)))
stopifnot(length(bc_called) == EXPECT$called, length(bc_singlet) == EXPECT$singlets)

In [ ]:
rna_clean <- rna_all[, bc_singlet, drop = FALSE]
adt_clean <- adt_all[, bc_singlet, drop = FALSE]

clean_adt_names <- function(x) {
  base  <- sub("[:|].*$", "", x)
  clone <- sub("^[^:|]*:?", "", sub("\\|.*$", "", x))
  ifelse(base %in% base[duplicated(base)] & nzchar(clone),
         paste0(base, "_", clone), base)
}
rownames(adt_clean) <- clean_adt_names(rownames(adt_clean))

cnt <- Matrix::rowSums(adt_clean)
det <- Matrix::rowMeans(adt_clean > 0)
absent <- names(cnt)[det < 0.01 | cnt < 1000]

cat("Dropping (never stained):\n")
print(data.frame(counts = cnt[absent], det_pct = round(100*det[absent], 3)))
adt_clean <- adt_clean[setdiff(rownames(adt_clean), absent), ]

rownames(adt_clean) <- sub("^([^:|]+):[^|]*", "\\1", rownames(adt_clean))
rownames(adt_clean) <- sub("^CD235a\\b", "CD235", rownames(adt_clean))
rownames(adt_clean) <- sub("_.*$", "", rownames(adt_clean))

stopifnot(identical(colnames(rna_clean), colnames(adt_clean)))

obj <- CreateSeuratObject(counts = rna_clean, assay = "RNA", project = "CordBlood")
obj[["ADT"]] <- CreateAssay5Object(counts = adt_clean)
rownames(obj[["ADT"]]) <- paste0("Protein-", rownames(obj[["ADT"]]))

md_tag <- calls[match(colnames(obj), calls$Cell_Index), ]
obj$SampleTag  <- md_tag$Sample_Tag
obj$Sample     <- md_tag$Sample_Name
obj$Donor      <- factor(obj$Sample)

print(obj)
rownames(obj[["ADT"]])

In [ ]:
FeatureScatter(obj, feature1 = "nCount_RNA", feature2 = "nFeature_RNA",
                     group.by = "Sample")
FeatureScatter(obj, feature1 = "nCount_ADT", feature2 = "nFeature_ADT",
                     group.by = "Sample") 

In [ ]:
before_n <- ncol(obj)

obj_QC <- subset(obj, subset = nFeature_ADT >10 &
                           nCount_ADT < 130000
                          )
FeatureScatter(obj_QC, feature1 = "nCount_RNA", feature2 = "nFeature_RNA",
                     group.by = "Sample")
FeatureScatter(obj_QC, feature1 = "nCount_ADT", feature2 = "nFeature_ADT",
                     group.by = "Sample") 


cat(sprintf("QC: %d -> %d cells (%.1f%% removed)\n",
            before_n, ncol(obj_QC), 100*(1-ncol(obj_QC)/before_n)))
print(table(obj$Sample))


In [ ]:
obj <- obj_QC

In [ ]:
DefaultAssay(obj) <- "RNA"
obj <- SCTransform(obj, assay = "RNA", method = "glmGamPoi",
                   vst.flavor = "v2", verbose = TRUE)


In [ ]:
DefaultAssay(obj) <- "ADT"
VariableFeatures(obj) <- rownames(obj[["ADT"]])      # few antibodies, take them all
obj <- NormalizeData(obj, assay = "ADT", normalization.method = "CLR", margin = 2,
                     verbose = FALSE)
obj <- ScaleData(obj, assay = "ADT", verbose = FALSE)

In [ ]:
obj <- RunPCA(obj, assay = "SCT", npcs = 40, reduction.name = "pca", verbose = FALSE)
obj <- RunPCA(obj, assay = "ADT", npcs = 40, reduction.name = "apca",
              features = rownames(obj[["ADT"]]), verbose = FALSE)

In [ ]:
RUN_HARMONY   <- FALSE 

if (RUN_HARMONY) {
  obj <- RunHarmony(obj, group.by.vars = "Sample", reduction = "pca",
                    assay.use = "SCT", reduction.save = "harmony_rna",
                    plot_convergence = FALSE, project.dim = FALSE)
  obj <- RunHarmony(obj, group.by.vars = "Sample", reduction = "apca",
                    assay.use = "ADT", reduction.save = "harmony_adt",
                    plot_convergence = FALSE, project.dim = FALSE)
  RED_RNA <- "harmony_rna"
  RED_ADT <- "harmony_adt"
} else {
  RED_RNA <- "pca"
  RED_ADT <- "apca"
}
cat("using reductions:", RED_RNA, "/", RED_ADT, "\n")

In [ ]:
ElbowPlot(obj, ndims = 40, reduction = RED_RNA) + ggtitle("RNA")
ElbowPlot(obj, ndims = 40, reduction = RED_ADT) + ggtitle("ADT")

In [ ]:
# fill these in from the elbows above
DIMS_RNA <- 1:29
DIMS_ADT <- 1:10

obj <- FindMultiModalNeighbors(obj,
        reduction.list = list(RED_RNA, RED_ADT),
        dims.list      = list(DIMS_RNA, DIMS_ADT),
        modality.weight.name = c("RNA.weight", "ADT.weight"))

obj <- RunUMAP(obj, nn.name = "weighted.nn",
               reduction.name = "wnn.umap", reduction.key = "wnnUMAP_", verbose = FALSE)

In [ ]:
obj <- FindClusters(obj, graph.name = "wsnn", algorithm = 3,
                    resolution = 0.3, verbose = FALSE)

obj <- RenameIdents(obj, 
                    "1"  = "0", 
                    "3"  = "0", 
                    "8"  = "0", 
                    "9"  = "0", 
                    "11" = "0", 
                    "13" = "0")

umap_coords <- Embeddings(obj, reduction = "wnn.umap")

target_cells <- rownames(umap_coords[umap_coords[, "wnnUMAP_1"] > -1 & 
                                     umap_coords[, "wnnUMAP_2"] > -4, ])

current_idents <- as.character(Idents(obj))
names(current_idents) <- colnames(obj)
current_idents[target_cells] <- "0"

Idents(obj) <- factor(current_idents)
obj$seurat_clusters <- Idents(obj)

table(Idents(obj))

DimPlot(obj, reduction = "wnn.umap", label = TRUE, repel = TRUE,
        pt.size = 0.3, label.size = 5)

In [ ]:
DimPlot(obj, reduction = "wnn.umap", group.by = "Sample", pt.size = 0.3)


In [ ]:
DefaultAssay(obj) <- "SCT"
obj <- PrepSCTFindMarkers(obj, verbose = FALSE)

markers_rna <- FindAllMarkers(obj, assay = "SCT", only.pos = TRUE,
                              min.pct = 0.25, logfc.threshold = 0.25,
                              test.use = "wilcox")

top_rna <- markers_rna %>% group_by(cluster) %>%
  slice_max(avg_log2FC, n = 30) %>% ungroup()
write.csv(markers_rna, file.path("markers_RNA_all.csv"), row.names = FALSE)


In [ ]:
DefaultAssay(obj) <- "ADT"
markers_adt <- FindAllMarkers(obj, assay = "ADT", only.pos = TRUE,
                              min.pct = 0.25, logfc.threshold = 0.25,
                              test.use = "wilcox")
write.csv(markers_adt, file.path("markers_ADT_all.csv"), row.names = FALSE)


In [ ]:
new_ids <- c(
  "0"  = "Erythroid cells",
  "2"  = "CD4 T cells",
  "4"  = "NK cells",
  "5"  = "Monocytes",
  "6"  = "CD8 T cells",
  "7"  = "B cells",
  "10" = "Regulatory T cells",
  "12" = "HSPC",
  "14" = "Dendritic cells"
)

obj <- RenameIdents(obj, new_ids)

cell_order <- c(
  "Erythroid cells", 
  "CD4 T cells", 
  "CD8 T cells", 
  "Regulatory T cells", 
  "B cells", 
  "NK cells", 
  "Monocytes", 
  "Dendritic cells", 
  "HSPC"
)

Idents(obj) <- factor(Idents(obj), levels = cell_order)

obj$CellType <- Idents(obj)

table(obj$CellType)

In [ ]:
saveRDS(obj, "Seurat_with_soup.rds")

In [ ]:
obj <- readRDS("./old/Seurat_with_soup.rds")

In [ ]:
library(Seurat)
library(ggplot2)
library(patchwork)
library(viridis)

# 1. SCT (RNA) markers
sct_markers <- unique(c(
  "SLC25A37", "SNCA", "ALAS2", "LGALS3",
  "TRAT1", "ICOS", "CD5", "BCL11B",
  "CD8B", "CD8A", "PASK", "KLRK1",
  "IL32", "KLRB1", "RGS1", "TRAC",
  "IGHD-membrane", "PAX5", "IGHM-secreted", "IGHM-membrane",
  "KLRF1", "NKG7", "PRF1", "IFNG",
  "CXCL1", "FCN1", "S100A9", "S100A12",
  "FCER1A", "JCHAIN", "CLEC10A", "CD1C",
  "KIT", "TYMS", "PDIA6", "GAPDH"
))

# ADT (Protein) markers
adt_markers <- unique(c(
  "Protein-CD235","Protein-CD71", "Protein-CD36",
  "Protein-CD4", "Protein-CD5",
  "Protein-CD8", "Protein-CD2",
  "Protein-CD25",
  "Protein-CD19", "Protein-CD79b",
  "Protein-CD56", "Protein-CD122",
  "Protein-CD64", "Protein-CD33",
  "Protein-HLA-DR",
  "Protein-CD34"
))

# 2. SCT DotPlot (Seurat default colors)
p_sct <- DotPlot(
  obj, 
  features = sct_markers, 
  assay = "SCT", 
  group.by = "CellType"
) + 
  RotatedAxis() + 
  ggtitle("SCT Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "plain", size = 12),
    axis.title.y = element_blank(),
    axis.title.x = element_blank(),
    axis.text.y = element_text(size = 10, face = "plain"),
    axis.text.x = element_text(size = 9, face = "plain"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 9, face = "plain")
  )

# 3. ADT DotPlot (Viridis color scale)
p_adt <- DotPlot(
  obj, 
  features = adt_markers, 
  assay = "ADT", 
  group.by = "CellType"
) + 
  scale_color_viridis_c() +
  RotatedAxis() + 
  ggtitle("ADT Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "plain", size = 12),
    axis.title.y = element_blank(),
    axis.text.y = element_blank(),
    axis.ticks.y = element_blank(),
    axis.title.x = element_blank(),
    axis.text.x = element_text(size = 9, face = "plain"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 9, face = "plain")
  )

# 4. Combine with adjusted width ratio (SCT: 1.33, ADT: 0.67)
multimodal_dotplot <- p_sct + p_adt + plot_layout(widths = c(1.25, 0.75), guides = "collect")

# 5. Save as high-resolution JPEG (600 DPI)
ggsave(
  "multimodal_dotplot_pre.jpeg", 
  plot = multimodal_dotplot, 
  device = "jpeg", 
  dpi = 600, 
  width = 18, 
  height = 6.2
)

In [ ]:
library(Seurat)
library(Matrix)

# ==============================================================================
# 1. LOAD MARKER CSV FILES & MAP CLUSTERS TO CELL TYPES
# ==============================================================================

# Read actual expression marker files
markers_rna <- read.csv("markers_RNA_all.csv", stringsAsFactors = FALSE)
markers_adt <- read.csv("markers_ADT_all.csv", stringsAsFactors = FALSE)

# Cluster-to-CellType mapping matching your Seurat object
cluster_map <- c(
  "0"  = "Erythroid cells",
  "2"  = "CD4 T cells",
  "4"  = "NK cells",
  "5"  = "Monocytes",
  "6"  = "CD8 T cells",
  "7"  = "B cells",
  "10" = "Regulatory T cells",
  "12" = "HSPC",
  "14" = "Dendritic cells"
)

# Map cluster IDs in CSVs to CellType names
markers_rna$CellType <- cluster_map[as.character(markers_rna$cluster)]
markers_adt$CellType <- cluster_map[as.character(markers_adt$cluster)]

# ==============================================================================
# 2. DYNAMICALLY BUILD ALLOW-LISTS FROM YOUR CSV DATA
# ==============================================================================

# Collect all allowed CellTypes for each gene and protein
rna_allow_list <- lapply(split(markers_rna$CellType, markers_rna$gene), unique)
adt_allow_list <- lapply(split(markers_adt$CellType, markers_adt$gene), unique)

message(sprintf("Generated dynamic allow-lists for %d RNA features and %d ADT features.", 
                length(rna_allow_list), length(adt_allow_list)))

# ==============================================================================
# 3. SEURAT V5 FEATURE CLEANUP FUNCTION
# ==============================================================================

#' Clean feature expression based on dynamic allow-lists in Seurat v5
clean_feature_counts <- function(seurat_obj, 
                                 assay = "RNA", 
                                 allow_list = list(), 
                                 idents_col = "CellType") {
  
  if (!assay %in% names(seurat_obj@assays)) {
    message(sprintf("Assay '%s' not found in object. Skipping.", assay))
    return(seurat_obj)
  }
  
  # Get cell type assignments for each barcode
  if (!is.null(idents_col) && idents_col %in% colnames(seurat_obj@meta.data)) {
    cell_idents <- as.character(seurat_obj[[idents_col]][, 1])
  } else {
    cell_idents <- as.character(Idents(seurat_obj))
  }
  names(cell_idents) <- colnames(seurat_obj)
  
  # Available layers and features in Seurat v5
  avail_layers <- Layers(seurat_obj[[assay]])
  assay_features <- rownames(seurat_obj[[assay]])
  
  # Process each layer (e.g., "counts", "data") in Seurat v5
  for (l in avail_layers) {
    mat <- GetAssayData(seurat_obj, assay = assay, layer = l)
    layer_modified <- FALSE
    
    for (feature in names(allow_list)) {
      if (feature %in% assay_features) {
        allowed_types <- allow_list[[feature]]
        disallowed_cells <- names(cell_idents)[!cell_idents %in% allowed_types]
        
        if (length(disallowed_cells) > 0) {
          mat[feature, disallowed_cells] <- 0
          layer_modified <- TRUE
        }
      }
    }
    
    if (layer_modified) {
      seurat_obj <- SetAssayData(seurat_obj, assay = assay, layer = l, new.data = mat)
    }
  }
  
  message(sprintf("Successfully cleaned '%s' assay across layers: %s", assay, paste(avail_layers, collapse = ", ")))
  return(seurat_obj)
}

# ==============================================================================
# 4. EXECUTE CLEANUP & RE-NORMALIZE ASSAYS
# ==============================================================================

# Clean RNA, SCT, and ADT assays using the dynamic CSV allow-lists
obj <- clean_feature_counts(obj, assay = "RNA", allow_list = rna_allow_list, idents_col = "CellType")
obj <- clean_feature_counts(obj, assay = "SCT", allow_list = rna_allow_list, idents_col = "CellType")
obj <- clean_feature_counts(obj, assay = "ADT", allow_list = adt_allow_list, idents_col = "CellType")

# Re-normalize RNA and ADT assays
obj <- NormalizeData(obj, assay = "RNA", normalization.method = "LogNormalize", verbose = FALSE)
obj <- NormalizeData(obj, assay = "ADT", normalization.method = "CLR", margin = 2, verbose = FALSE)

In [ ]:
library(Seurat)
library(ggplot2)
library(patchwork)
library(viridis)

# 1. SCT (RNA) markers
sct_markers <- unique(c(
  "SLC25A37", "SNCA", "ALAS2", "LGALS3",
  "TRAT1", "ICOS", "CD5", "BCL11B",
  "CD8B", "CD8A", "PASK", "KLRK1",
  "IL32", "KLRB1", "RGS1", "TRAC",
  "IGHD-membrane", "PAX5", "IGHM-secreted", "IGHM-membrane",
  "KLRF1", "NKG7", "PRF1", "IFNG",
  "CXCL1", "FCN1", "S100A9", "S100A12",
  "FCER1A", "JCHAIN", "CLEC10A", "CD1C",
  "KIT", "TYMS", "PDIA6", "GAPDH"
))

# ADT (Protein) markers
adt_markers <- unique(c(
  "Protein-CD235","Protein-CD71", "Protein-CD36",
  "Protein-CD4", "Protein-CD5",
  "Protein-CD8", "Protein-CD2",
  "Protein-CD25",
  "Protein-CD19", "Protein-CD79b",
  "Protein-CD56", "Protein-CD122",
  "Protein-CD64", "Protein-CD33",
  "Protein-HLA-DR",
  "Protein-CD34"
))

# 2. SCT DotPlot (Seurat default colors)
p_sct <- DotPlot(
  obj, 
  features = sct_markers, 
  assay = "SCT", 
  group.by = "CellType"
) + 
  RotatedAxis() + 
  ggtitle("SCT Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "plain", size = 12),
    axis.title.y = element_blank(),
    axis.title.x = element_blank(),
    axis.text.y = element_text(size = 10, face = "plain"),
    axis.text.x = element_text(size = 9, face = "plain"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 9, face = "plain")
  )

# 3. ADT DotPlot (Viridis color scale)
p_adt <- DotPlot(
  obj, 
  features = adt_markers, 
  assay = "ADT", 
  group.by = "CellType"
) + 
  scale_color_viridis_c() +
  RotatedAxis() + 
  ggtitle("ADT Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "plain", size = 12),
    axis.title.y = element_blank(),
    axis.text.y = element_blank(),
    axis.ticks.y = element_blank(),
    axis.title.x = element_blank(),
    axis.text.x = element_text(size = 9, face = "plain"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 9, face = "plain")
  )

# 4. Combine with adjusted width ratio (SCT: 1.33, ADT: 0.67)
multimodal_dotplot <- p_sct + p_adt + plot_layout(widths = c(1.25, 0.75), guides = "collect")

# 5. Save as high-resolution JPEG (600 DPI)
ggsave(
  "multimodal_dotplot.jpeg", 
  plot = multimodal_dotplot, 
  device = "jpeg", 
  dpi = 600, 
  width = 18, 
  height = 6.2
)

In [ ]:
saveRDS(obj, "Seurat.rds")

sessionInfo()

## Analysis

In [ ]:
obj <-readRDS("Seurat.rds")
obj
gc()

In [ ]:
library(ggplot2)

In [ ]:
DimPlot(obj, reduction = "wnn.umap", pt.size = 1, group.by = "Sample") + 
  ggtitle(NULL)
ggsave("dimplot_sample.jpeg", plot = last_plot(), device = "jpeg", dpi = 600, width = 10.5, height = 9)


In [ ]:
DimPlot(obj, reduction = "wnn.umap", pt.size = 1, label = TRUE,  group.by = "CellType") + 
  ggtitle(NULL)
ggsave("dimplot_celltype.jpeg", plot = last_plot(), device = "jpeg", dpi = 600, width = 10.8, height = 9)

In [ ]:
Features(obj[["ADT"]])

In [ ]:
library(patchwork)
library(viridis)

sct_markers <- unique(c(
  "ALAS2","SLC25A37", "SNCA", "LGALS3",
  "TRAT1", "ICOS", "CD5", "BCL11B",
  "CD8B", "CD8A", "PASK", "KLRK1",
  "IL32", "KLRB1", "RGS1", "TRAC",
  "IGHD-membrane", "PAX5", "IGHM-secreted", "IGHM-membrane",
  "KLRF1", "NKG7", "PRF1", "IFNG",
  "CXCL1", "FCN1", "S100A9", "S100A12",
  "FCER1A", "JCHAIN", "CLEC10A", "CD1C",
  "KIT", "TYMS", "PDIA6", "GAPDH"
))

adt_markers <- unique(c(
  "Protein-CD235","Protein-CD71", "Protein-CD36",
  "Protein-CD3","Protein-CD4", "Protein-CD8", 
  "Protein-CD45RA","Protein-CD45RO",  
  "Protein-CD25","Protein-CD62L","Protein-CCR7",
  "Protein-CD19", "Protein-CD79b",
  "Protein-CD56", "Protein-CD122",
  "Protein-CD14", "Protein-CD64",
  "Protein-CD86","Protein-HLA-DR",
  "Protein-CD34","Protein-CD117"
))

p_sct <- DotPlot(
  obj, 
  features = sct_markers, 
  assay = "SCT", 
  group.by = "CellType"
) + 
  RotatedAxis() + 
  ggtitle("Gene Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "plain", size = 12),
    axis.title.y = element_blank(),
    axis.title.x = element_blank(),
    axis.text.y = element_text(size = 10, face = "plain"),
    axis.text.x = element_text(size = 9, face = "plain"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 9, face = "plain")
  )

p_adt <- DotPlot(
  obj, 
  features = adt_markers, 
  assay = "ADT", 
  group.by = "CellType"
) + 
  scale_color_viridis_c() +
  RotatedAxis() + 
  ggtitle("Protein Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "plain", size = 12),
    axis.title.y = element_blank(),
    axis.text.y = element_blank(),
    axis.ticks.y = element_blank(),
    axis.title.x = element_blank(),
    axis.text.x = element_text(size = 9, face = "plain"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 9, face = "plain")
  )

multimodal_dotplot <- p_sct + p_adt + plot_layout(widths = c(1.4, 0.9), guides = "collect")

multimodal_dotplot

ggsave(
  "multimodal_dotplot.jpeg", 
  plot = multimodal_dotplot, 
  device = "jpeg", 
  dpi = 600, 
  width = 18.5 , 
  height = 6.2
)

In [ ]:
library(viridis)

adt_features <- rownames(obj[["ADT"]])

avg_adt <- AverageExpression(obj, assays = "ADT", features = adt_features, group.by = "CellType")$ADT

celltype_hc <- hclust(dist(t(avg_adt)))            # Distance between cell types
adt_hc      <- hclust(dist(avg_adt))               # Distance between ADT features

ordered_celltypes <- colnames(avg_adt)[celltype_hc$order]
ordered_adts      <- rownames(avg_adt)[adt_hc$order]

obj$CellType_Clustered <- factor(obj$CellType, levels = ordered_celltypes)

p_adt_clustered <- DotPlot(
  obj, 
  features = ordered_adts, 
  assay = "ADT", 
  group.by = "CellType_Clustered"
) + 
  scale_color_viridis_c() +
  RotatedAxis() + 
  ggtitle("Protein Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "plain", size = 12),
    axis.title.y = element_blank(),
    axis.title.x = element_blank(),
    axis.text.y = element_text(size = 10, face = "plain"),
    axis.text.x = element_text(size = 9, face = "plain"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 9, face = "plain")
  ) + coord_flip()

p_adt_clustered

ggsave(
  "adt_clustered_dotplot.jpeg", 
  plot = p_adt_clustered, 
  device = "jpeg", 
  dpi = 600, 
  width = 5.6, 
  height = 12
)

In [ ]:
sct_markers <- unique(c(
  "ALAS2","SLC25A37", "SNCA", "LGALS3",
  "TRAT1", "ICOS", "CD5", "BCL11B",
  "CD8B", "CD8A", "PASK", "KLRK1",
  "IL32", "KLRB1", "RGS1", "TRAC",
  "IGHD-membrane", "PAX5", "IGHM-secreted", "IGHM-membrane",
  "KLRF1", "NKG7", "PRF1", "IFNG",
  "CXCL1", "FCN1", "S100A9", "S100A12",
  "FCER1A", "JCHAIN", "CLEC10A", "CD1C",
  "KIT", "TYMS", "PDIA6", "GAPDH"
))

p_gex_only <- DotPlot(
  obj, 
  features = sct_markers, 
  assay = "SCT", 
  group.by = "CellType"
) + 
  # scale_color_viridis_c() +
  RotatedAxis() + 
  ggtitle("Gene Expression") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "plain", size = 12),
    axis.title.y = element_blank(),
    axis.title.x = element_blank(),
    axis.text.y = element_text(size = 10, face = "plain"),
    axis.text.x = element_text(size = 9, face = "plain"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 9, face = "plain")
  )

ggsave(
  "gex_only_dotplot.jpeg", 
  plot = p_gex_only, 
  device = "jpeg", 
  dpi = 600, 
  width = 14, 
  height = 5
)

In [ ]:
fp <- FeaturePlot(
  object = obj, 
  features = c(adt_markers,"Protein-CD38","Protein-CD40","Protein-CD49d"), 
  reduction = "wnn.umap", 
  pt.size = 0.5,
  ncol = 6
) 

ggsave(
  filename = "adt_feature_plots.jpeg", 
  plot = fp, 
  device = "jpeg", 
  dpi = 600, 
  width = 32, 
  height = 20
)


In [ ]:
fp <- FeaturePlot(
  object = obj, 
  features = c(adt_markers), 
  reduction = "wnn.umap", 
  pt.size = 0.5,
  ncol = 3
) 

ggsave(
  filename = "adt_feature_plots_vertical.jpeg", 
  plot = fp, 
  device = "jpeg", 
  dpi = 600, 
  width = 18, 
  height = 32
)


In [ ]:
library(dplyr)
library(ggalluvial)

plot_data <- obj@meta.data %>%
  group_by(Sample, CellType) %>%
  summarise(Count = n(), .groups = "drop") %>%
  group_by(Sample) %>%
  mutate(Proportion = Count / sum(Count))

plot_data$CellType <- factor(plot_data$CellType, levels = levels(obj$CellType))

erythroid_order <- plot_data %>%
  filter(CellType == "Erythroid cells") %>%
  arrange(desc(Proportion)) %>%
  pull(Sample)

sample_levels <- c(erythroid_order, setdiff(unique(plot_data$Sample), erythroid_order))
plot_data$Sample <- factor(plot_data$Sample, levels = sample_levels)

p_alluvial <- ggplot(plot_data, 
                     aes(x = Sample, 
                         y = Proportion, 
                         alluvium = CellType, 
                         stratum = CellType, 
                         fill = CellType)) +
  geom_flow(alpha = 0.6, color = "black") +
  geom_stratum(alpha = 0.9, color = "black") +
  scale_y_continuous(labels = scales::percent_format(), expand = c(0, 0)) +
  theme_minimal() +
  labs(x = "Sample", y = "Cell Proportion", fill = "Cell Type") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 12, face = "plain"),
    axis.text.y = element_text(size = 10, face = "plain"),
    axis.title = element_text(size = 12, face = "plain"),
    legend.text = element_text(size = 10, face = "plain"),
    legend.title = element_text(size = 11, face = "plain"),
    panel.grid.major.x = element_blank()
  )

p_alluvial

ggsave(
  "Alluvial_per_sample_proportions.png",
  plot   = p_alluvial,
  width  = 27,
  height = 15,
  units  = "cm",
  dpi    = 600,
  bg     = "white"
)

In [ ]:
Eb <- readRDS("Eb.rds")
# obj <- readRDS("Seurat.rds")
obj <- readRDS("./Old/Seurat_with_soup.rds")

In [ ]:
# 1. Extract and map annotations
eb_celltypes <- as.character(Eb$CellType)
names(eb_celltypes) <- colnames(Eb)

updated_celltypes <- as.character(obj$CellType)
names(updated_celltypes) <- colnames(obj)

common_cells <- intersect(colnames(obj), names(eb_celltypes))
updated_celltypes[common_cells] <- eb_celltypes[common_cells]

# 2. Subset object in-place to keep ONLY cells with valid, non-NA annotations
valid_cells <- names(updated_celltypes)[!is.na(updated_celltypes) & updated_celltypes != "NA"]
obj <- subset(obj, cells = valid_cells)

# 3. Filter updated vector for the remaining valid cells
updated_celltypes_clean <- updated_celltypes[valid_cells]

# 4. Define target order & intersect with levels present to prevent accidental factor NAs
cell_order <- c(
  "CFU-E", 
  "Pro Eb", 
  "Baso Eb", 
  "Poly Eb",
  "Ortho Eb", 
  "ARG1+ Ortho Eb",
  "CD4 T cells", 
  "CD8 T cells", 
  "Regulatory T cells",
  "B cells", 
  "NK cells", 
  "Monocytes", 
  "Dendritic cells", 
  "HSPC"
)

actual_levels <- intersect(cell_order, unique(updated_celltypes_clean))

# 5. Convert to clean factor and set Idents in-place
obj$CellType <- factor(updated_celltypes_clean, levels = actual_levels)
Idents(obj) <- obj$CellType

In [ ]:
table(obj@meta.data$CellType)

In [ ]:
saveRDS(obj, "Seurat_CellChat.rds")

In [ ]:
DimPlot(obj, reduction = "wnn.umap", pt.size = 0.5, label = TRUE,  group.by = "CellType") + 
  ggtitle(NULL)
ggsave("deeper_dimplot_celltype.jpeg", plot = last_plot(), device = "jpeg", dpi = 600, width = 10.8, height = 9)

In [ ]:
plot_data <- obj@meta.data %>%
  group_by(Sample, CellType) %>%
  summarise(Count = n(), .groups = "drop") %>%
  group_by(Sample) %>%
  mutate(Percentage = (Count / sum(Count)) * 100)

plot_data_low <- plot_data %>%
  group_by(CellType) %>%
  filter(max(Percentage) <= 25) %>%
  ungroup()

remaining_levels <- intersect(levels(obj$CellType), unique(plot_data_low$CellType))
plot_data_low$CellType <- factor(plot_data_low$CellType, levels = remaining_levels)

p_low_boxplot <- ggplot(plot_data_low, aes(x = CellType, y = Percentage, fill = CellType)) +
  # ADD THIS LAYER FIRST: This creates the "T" shaped whisker caps
  stat_boxplot(geom = "errorbar", width = 0.25, color = "black", size = 0.5) +
  
  # Standard boxplot layer (hiding default outliers since we add raw points next)
  geom_boxplot(width = 0.6, color = "black", size = 0.5, alpha = 0.7, outlier.shape = NA) +
  
  # Sample data points spread out slightly across the boxes
  geom_jitter(width = 0.15, size = 1.8, alpha = 0.7, color = "black") +
  
  # Force Y-axis to focus cleanly between 0% and 25%
  scale_y_continuous(labels = function(x) paste0(x, "%"), limits = c(0, 25)) +
  theme_minimal() +
  labs(x = "Cell Type", y = "Cell Percentage per Sample") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 11, face = "plain"),
    axis.text.y = element_text(size = 10, face = "plain"),
    axis.title = element_text(size = 12, face = "plain"),
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank(),
    # Seurat Style Legend overrides:
    legend.title = element_blank(), 
    legend.text = element_text(size = 10),
    legend.background = element_blank(),
    legend.key = element_blank()
  ) +
  # Recreates Seurat's circular dot legend keys
  guides(fill = guide_legend(override.aes = list(shape = 21, size = 4, alpha = 1, color = NA)))

p_low_boxplot

ggsave(
  "Abundance_Boxplot.png",
  plot   = p_low_boxplot,
  width  = 30,
  height = 15,
  units  = "cm",
  dpi    = 600,
  bg     = "white"
)


## pySCENIC

In [ ]:
# Export for pySCENIC

library(magrittr)
library(SingleCellExperiment)
library(SCopeLoomR)

exprMat <- obj[["SCT"]]$data
cellInfo <- obj@meta.data

min_cells <- 0
loci1 <- which(rowSums(exprMat > 0) >= min_cells)

exprMat_filter <- exprMat[loci1, ]
head(exprMat_filter)

add_cell_annotation <- function(loom, cellAnnotation)
{
  cellAnnotation <- data.frame(cellAnnotation)
  if(any(c("nGene", "nUMI") %in% colnames(cellAnnotation)))
  {
    warning("Columns 'nGene' and 'nUMI' will not be added as annotations to the loom file.")
    cellAnnotation <- cellAnnotation[,colnames(cellAnnotation) != "nGene", drop=FALSE]
    cellAnnotation <- cellAnnotation[,colnames(cellAnnotation) != "nUMI", drop=FALSE]
  }
  
  if(ncol(cellAnnotation)<=0) stop("The cell annotation contains no columns")
  if(!all(get_cell_ids(loom) %in% rownames(cellAnnotation))) stop("Cell IDs are missing in the annotation")
  
  cellAnnotation <- cellAnnotation[get_cell_ids(loom),,drop=FALSE]
  for(cn in colnames(cellAnnotation))
  {
    add_col_attr(loom=loom, key=cn, value=cellAnnotation[,cn])
  }
  
  invisible(loom)
}

loom <- build_loom("./pySCENIC/Seurat.loom", dgem=exprMat_filter)
loom <- add_cell_annotation(loom, cellInfo)
close_loom(loom)

DefaultAssay(obj) <- "SCT"


In [2]:
obj <- readRDS("Seurat.rds")

In [3]:
head(obj@meta.data)

,orig.ident,nCount_RNA,nFeature_RNA,nCount_ADT,nFeature_ADT,SampleTag,Sample,Donor,nCount_SCT,nFeature_SCT,RNA.weight,ADT.weight,wsnn_res.0.3,seurat_clusters,CellType
,<fct>,<dbl>,<int>,<dbl>,<int>,<chr>,<chr>,<fct>,<dbl>,<int>,<dbl>,<dbl>,<fct>,<fct>,<fct>
5749,CordBlood,120,25,2108,34,SampleTag06_hs,Cord_Blood_6,Cord_Blood_6,102,32,0.22866400,0.7713360,10,10,Regulatory T cells
7720,CordBlood,159,42,12207,36,SampleTag06_hs,Cord_Blood_6,Cord_Blood_6,98,34,0.08448437,0.9155156,2,2,CD4 T cells
8889,CordBlood,47,28,833,34,SampleTag01_hs,Cord_Blood_1,Cord_Blood_1,73,33,0.56345778,0.4365422,4,4,NK cells
10649,CordBlood,159,46,2937,36,SampleTag03_hs,Cord_Blood_3,Cord_Blood_3,111,56,0.08877805,0.9112220,6,6,CD8 T cells
11580,CordBlood,54,5,3694,35,SampleTag01_hs,Cord_Blood_1,Cord_Blood_1,93,10,0.15882343,0.8411766,3,0,Erythroid cells
13268,CordBlood,253,59,2541,34,SampleTag01_hs,Cord_Blood_1,Cord_Blood_1,101,43,0.40611880,0.5938812,4,4,NK cells


In [4]:
auc_raw <- t(read.csv("./pySCENIC/pySCENIC-AUC-Raw.csv", row.names = 1, check.names = FALSE))
auc_bin <- t(read.csv("./pySCENIC/pySCENIC-AUC-Binary.csv", row.names = 1, check.names = FALSE))

rownames(auc_raw) <- gsub("\\s*\\(\\+\\)", "", rownames(auc_raw))
rownames(auc_bin) <- gsub("\\s*\\(\\+\\)", "", rownames(auc_bin))

obj[["AUC"]] <- CreateAssayObject(counts = auc_raw)
obj[["AUC_Binary"]] <- CreateAssayObject(counts = auc_bin)

In [9]:
TFs <- rownames(obj[["AUC"]])
print(TFs)

 [1] "BACH2"  "BCL6"   "EGR1"   "FOSB"   "FOXO1"  "FOXP1"  "IKZF1"  "IRF4"  
 [9] "IRF8"   "JUN"    "JUNB"   "MYC"    "PAX5"   "PRDM1"  "RUNX3"  "STAT1" 
[17] "STAT3"  "STAT4"  "STAT5A" "TBX21" 


In [5]:
Eb <- readRDS("Eb.rds")

# 1. Extract and map annotations
eb_celltypes <- as.character(Eb$CellType)
names(eb_celltypes) <- colnames(Eb)

updated_celltypes <- as.character(obj$CellType)
names(updated_celltypes) <- colnames(obj)

common_cells <- intersect(colnames(obj), names(eb_celltypes))
updated_celltypes[common_cells] <- eb_celltypes[common_cells]

# 2. Subset object in-place to keep ONLY cells with valid, non-NA annotations
valid_cells <- names(updated_celltypes)[!is.na(updated_celltypes) & updated_celltypes != "NA"]
obj <- subset(obj, cells = valid_cells)

# 3. Filter updated vector for the remaining valid cells
updated_celltypes_clean <- updated_celltypes[valid_cells]

# 4. Define target order & intersect with levels present to prevent accidental factor NAs
cell_order <- c(
  "CFU-E", 
  "Pro Eb", 
  "Baso Eb", 
  "Poly Eb",
  "Ortho Eb", 
  "ARG1+ Ortho Eb",
  "CD4 T cells", 
  "CD8 T cells", 
  "Regulatory T cells",
  "B cells", 
  "NK cells", 
  "Monocytes", 
  "Dendritic cells", 
  "HSPC"
)

actual_levels <- intersect(cell_order, unique(updated_celltypes_clean))

# 5. Convert to clean factor and set Idents in-place
obj$CellType <- factor(updated_celltypes_clean, levels = actual_levels)
Idents(obj) <- obj$CellType

In [6]:
head(obj@meta.data)

,orig.ident,nCount_RNA,nFeature_RNA,nCount_ADT,nFeature_ADT,SampleTag,Sample,Donor,nCount_SCT,nFeature_SCT,RNA.weight,ADT.weight,wsnn_res.0.3,seurat_clusters,CellType,nCount_AUC,nFeature_AUC,nCount_AUC_Binary,nFeature_AUC_Binary
,<fct>,<dbl>,<int>,<dbl>,<int>,<chr>,<chr>,<fct>,<dbl>,<int>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<dbl>,<int>,<dbl>,<int>
5749,CordBlood,120,25,2108,34,SampleTag06_hs,Cord_Blood_6,Cord_Blood_6,102,32,0.22866400,0.7713360,10,10,Regulatory T cells,1.2702215,20,6,6
7720,CordBlood,159,42,12207,36,SampleTag06_hs,Cord_Blood_6,Cord_Blood_6,98,34,0.08448437,0.9155156,2,2,CD4 T cells,0.9020526,19,2,2
8889,CordBlood,47,28,833,34,SampleTag01_hs,Cord_Blood_1,Cord_Blood_1,73,33,0.56345778,0.4365422,4,4,NK cells,1.0779222,19,4,4
10649,CordBlood,159,46,2937,36,SampleTag03_hs,Cord_Blood_3,Cord_Blood_3,111,56,0.08877805,0.9112220,6,6,CD8 T cells,1.2792962,19,8,8
11580,CordBlood,54,5,3694,35,SampleTag01_hs,Cord_Blood_1,Cord_Blood_1,93,10,0.15882343,0.8411766,3,0,Baso Eb,0.6681292,18,2,2
13268,CordBlood,253,59,2541,34,SampleTag01_hs,Cord_Blood_1,Cord_Blood_1,101,43,0.40611880,0.5938812,4,4,NK cells,1.2272108,20,6,6


In [7]:
saveRDS(obj,"Seurat_pySCENIC.rds")

In [8]:
library(Seurat); library(Matrix)

obj <- readRDS("Seurat_pySCENIC.rds")     # SCT + ADT + AUC + AUC_Binary
OUT <- "Vitessce"; dir.create(OUT, showWarnings = FALSE, recursive = TRUE)

getm <- function(assay, layer) tryCatch(
  GetAssayData(obj, assay = assay, layer = layer),
  error = function(e) GetAssayData(obj, assay = assay, slot = layer))

rna  <- getm("SCT", "data")                         # log-norm RNA
adt  <- getm("ADT", "data")                         # CLR-norm белок
auc  <- getm("AUC",        "counts")                # raw regulon AUC
aucb <- getm("AUC_Binary", "counts")                # binary regulon (0/1)

bc <- colnames(rna)                                 
stopifnot(all(bc %in% colnames(adt)),
          all(bc %in% colnames(auc)),
          all(bc %in% colnames(aucb)))
adt  <- adt[,  bc, drop = FALSE]
auc  <- auc[,  bc, drop = FALSE]
aucb <- aucb[, bc, drop = FALSE]

rownames(aucb) <- paste0("Binary ", sub("\\(\\+\\)$", "", rownames(aucb)))

writeMtx <- function(m, f) Matrix::writeMM(as(m, "dgCMatrix"), file.path(OUT, f))
writeMtx(rna,  "rna.mtx");        writeLines(rownames(rna),  file.path(OUT, "rna_features.txt"))
writeMtx(adt,  "adt.mtx");        writeLines(rownames(adt),  file.path(OUT, "adt_features.txt"))
writeMtx(auc,  "auc.mtx");        writeLines(rownames(auc),  file.path(OUT, "auc_features.txt"))
writeMtx(aucb, "auc_binary.mtx"); writeLines(rownames(aucb), file.path(OUT, "auc_binary_features.txt"))
writeLines(bc, file.path(OUT, "barcodes.txt"))

um  <- Embeddings(obj, "wnn.umap")[bc, ]
obs <- data.frame(
  barcode  = bc,
  CellType = as.character(obj$CellType[bc]),
  Sample   = as.character(obj$Sample[bc]),
  UMAP_1   = um[, 1], UMAP_2 = um[, 2],
  check.names = FALSE
)
write.csv(obs, file.path(OUT, "obs.csv"), row.names = FALSE)

cat(sprintf("exported %d cells | RNA %d | ADT %d | REG %d | REGb %d\n",
            length(bc), nrow(rna), nrow(adt), nrow(auc), nrow(aucb)))

NULL

NULL

NULL

NULL

exported 22061 cells | RNA 347 | ADT 37 | REG 20 | REGb 20


In [ ]:
if (!requireNamespace("pheatmap", quietly = TRUE))
  install.packages("pheatmap", repos = "https://cloud.r-project.org")
library(pheatmap)

# Mean regulon AUC per broad cell type; clean regulon names (Gata1(+) / Gata1... -> Gata1)
auc_mat <- as.matrix(GetAssayData(obj, assay = "AUC", layer = "counts"))
rownames(auc_mat) <- gsub("[^[:alnum:]]", "", rownames(auc_mat))
ct  <- obj$CellType
grp <- if (is.factor(ct)) levels(ct) else sort(unique(as.character(ct)))
auc_avg <- sapply(grp, function(g) rowMeans(auc_mat[, ct == g, drop = FALSE]))
auc_avg <- auc_avg[apply(auc_avg, 1, sd) > 0, , drop = FALSE]

bwr <- colorRampPalette(c("blue", "white", "red"))(100)

# TF (row) dendrogram + clustering; cell types kept in atlas order
p_all <- pheatmap(auc_avg, scale = "row", color = bwr,
                  cluster_rows = TRUE, cluster_cols = TRUE, treeheight_row = 35,
                  border_color = NA, fontsize_row = 10, fontsize_col = 10,
                  # main = "pySCENIC regulon activity (mean AUC)", 
                  silent = TRUE)

jpeg("Regulons_AllCellTypes.jpeg", width = 4.9, height = 9, units = "in", res = 600, quality = 95)
grid::grid.newpage(); grid::grid.draw(p_all$gtable); dev.off()

grid::grid.newpage(); grid::grid.draw(p_all$gtable)